In [ ]:
import cv2
import einops
import matplotlib.pyplot as plt
import mediapy
import numpy as np

import jax.numpy as jnp

from openpi.policies.libero_reason_dataset import LiberoSkillReasonDataset
from openpi.training import config as _config

In [ ]:
data_config = _config.get_config('pi05_libero_skill_reason_lora_v2')
dataset = LiberoSkillReasonDataset(data_config.data.base_config, data_config.model.action_horizon)

In [ ]:
import os
from pathlib import Path
import sys
SCRIPT_DIR = Path("../py_script")
sys.path.append(str(SCRIPT_DIR))
from vlm_interfaces import *

In [ ]:
from vla_verify.scene_graph import TaskSceneGraph
PDDL_PATH = SCRIPT_DIR / "pddl" / "pick_place_domain.pddl"
pddl_domain_text = open(PDDL_PATH).read()

llm_interface, vlm_interface = get_openrouter_interfaces()

scene_graph = TaskSceneGraph(pddl_domain_text, vlm_interface)

In [ ]:
def image_tensor_to_cv2(image, resolution=(512,512)):
    return cv2.resize(np.array(einops.rearrange(image, "c h w -> h w c") * 255, dtype=np.uint8), resolution, interpolation=cv2.INTER_LANCZOS4)

def get_episode(episode_idx):
    reasonings = dataset.reasoning[episode_idx]
    start_idx = dataset.episode_starts[episode_idx]
    end_idx = dataset.episode_ends[episode_idx]
    data = dataset.hf_dataset[int(start_idx)]
    video_frames = []
    for i in range(start_idx, end_idx):
        img_data = dataset.hf_dataset[i]['image']
        video_frames.append(image_tensor_to_cv2(img_data))
    return reasonings, video_frames

reasonings, video_frames = get_episode(439)
mediapy.write_video(f'sample.mp4', video_frames, fps=20)

In [ ]:
def _nl_task_to_pddl(llm_response, avail_actions

def process_episode(episode):
    reasonings, video_frames = episode
    scene_graph.read_image(video_frames, hint=f"The robot is trying to {reasonings['segments'][0]['instruction']}", ground=True)
    # results = scene_graph.ground_video(additional_points_labels=[
    #     ("robot", [[255, 90]])
    # ])
    for segment in reasonings['segments'][1:]:
        pass


In [ ]:
process_episode((reasonings, video_frames))

In [ ]:
print(scene_graph.simulator)

In [ ]:
import inspect
print(inspect.getsource(scene_graph._read_image))